In [805]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pandas as pd
pd.set_option('display.max_columns', 200)

In [806]:
df = pd.read_csv('../data/kaggle_b2_fraud_train_v4.csv')
df_test = pd.read_csv('../data/kaggle_b2_fraud_test_v4.csv')

## A. Qualité des données / nettoyage logique

### 1.1 Valeurs impossibles détectées

In [807]:
# suppression des âges négatifs dans df
df = df[df['age'] >= 0]

# remplacement des âges négatifs par 0 dans df_test
df_test.loc[df_test['age'] < 0, 'age'] = 0


# suppression des dates de création de compte négatives dans df
df = df[df["tenure_months"] >= 0]

# remplacement des tenure négatifs par 0 dans df_test
df_test.loc[df_test["tenure_months"] < 0, "tenure_months"] = 0


# suppression des revenus négatifs dans df
df = df[df["annual_income_eur"] >= 0]

# remplacement des revenus négatifs par 0 dans df_test
df_test.loc[df_test["annual_income_eur"] < 0, "annual_income_eur"] = 0


# suppression des montants négatifs dans df
df = df[df["avg_amount_30d_eur"] >= 0]

# remplacement des montants négatifs par 0 dans df_test
df_test.loc[df_test["avg_amount_30d_eur"] < 0, "avg_amount_30d_eur"] = 0

### 1.2 Corrections de typologie nécessaires

In [808]:
# Conversions de types
for dataset in [df, df_test]:
    dataset['is_new_device'] = dataset['is_new_device'].astype('int64')
    dataset['days_since_last_login'] = dataset['days_since_last_login'].astype('int64')

### 1.3 suppression doublons inutiles

In [809]:
# suppression des lignes avec des customer_id en double
df = df.drop_duplicates(subset=['customer_id'], keep='first')
df_test = df_test.drop_duplicates(subset=['customer_id'], keep='first')

### 1.4 doublons stricts

In [810]:
df = df.drop_duplicates()
df_test = df_test.drop_duplicates()

## B. Data leakage connu

In [811]:
"""colonnes_a_supprimer = [
    'chargeback_resolution_time_days',
    'post_event_status_code'
]

df = df.drop(columns=colonnes_a_supprimer)
df_test = df_test.drop(columns=colonnes_a_supprimer)"""

"colonnes_a_supprimer = [\n    'chargeback_resolution_time_days',\n    'post_event_status_code'\n]\n\ndf = df.drop(columns=colonnes_a_supprimer)\ndf_test = df_test.drop(columns=colonnes_a_supprimer)"

## C. Identifiants et colonnes inutiles

In [812]:
# Suppression des colonnes avec trop de NaN (>90%)
cols_to_drop = ["partner_risk_indicator"]

df = df.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

In [813]:
# suppression de 680 lignes avec des customer_id en double ( très peu de changement donc variation)
df = df.drop_duplicates(subset=['customer_id'], keep='first')
df_test = df_test.drop_duplicates(subset=['customer_id'], keep='first')

In [815]:
# trop granulaire, inutile ou non exploitable
cols_to_drop = [
    "referrer_code",   # trop granulaire     # trop granulaire : risque overfitting
    "account_id",      # inutile
    "signup_date",      # pas d'informations facilement exploitable            # trop granulaire
]

df = df.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

In [816]:
# Créer la colonne has_second_email (1 si secondary_email existe, 0 sinon)
df['has_second_email'] = df['secondary_email'].notna().astype(int)
df_test['has_second_email'] = df_test['secondary_email'].notna().astype(int)

# Supprimer la colonne secondary_email
df = df.drop(columns=['secondary_email'])
df_test = df_test.drop(columns=['secondary_email'])

In [818]:
# Créer les flags pour TOUTES les variables avec missingness informatif
for dataset in [df, df_test]:
    dataset['is_missing_max_amount_30d_eur'] = dataset['max_amount_30d_eur'].isna().astype(int)

In [820]:
## supprimer les colonnes de texte car on sait pas les traiter
"""text_cols = ["customer_note","last_ticket_subject"]

df = df.drop(columns=text_cols)
df_test = df_test.drop(columns=text_cols)"""

'text_cols = ["customer_note","last_ticket_subject"]\n\ndf = df.drop(columns=text_cols)\ndf_test = df_test.drop(columns=text_cols)'

In [821]:
df = df.drop(columns=[
    'internal_signal_1',
    'internal_signal_2',
    'internal_signal_3',
    'internal_signal_4',
    'internal_signal_5',
    'internal_signal_6',
    'internal_signal_7',
    'internal_signal_8'
])

## D. Création de features basiques

In [822]:
# création de combinaison nouvelles et logiques

for dataset in [df, df_test]:
    # Interaction binaire × numérique
    dataset["is_new_device_x_num_devices"] = dataset["is_new_device"] * dataset["num_devices_30d"]

    # Interaction binaire × binaire
    dataset["is_vpn_x_ip_risk"] = dataset["is_vpn"] * dataset["ip_risk_z"]

## E. imputations ne dépendant pas de statistiques

In [823]:
df['max_amount_30d_eur'] = df['max_amount_30d_eur'].fillna(0)
df_test['max_amount_30d_eur'] = df_test['max_amount_30d_eur'].fillna(0)

In [824]:
df["ip_risk_z"] = df["ip_risk_z"].fillna(0)
df_test["ip_risk_z"] = df_test["ip_risk_z"].fillna(0)

## F : Nettoyage des variables catégorielles

In [826]:
## 
"""df.occupation.unique()

mapping = {
    'self employed': 'self_employed',
    'free lancer': 'freelancer',
    'freeelancer': 'freelancer',
    'public-sector': 'public_sector',
}

df['occupation'] = df['occupation'].str.strip().str.lower().replace(mapping)"""

"df.occupation.unique()\n\nmapping = {\n    'self employed': 'self_employed',\n    'free lancer': 'freelancer',\n    'freeelancer': 'freelancer',\n    'public-sector': 'public_sector',\n}\n\ndf['occupation'] = df['occupation'].str.strip().str.lower().replace(mapping)"

In [828]:
"""df.payment_method.unique()

mapping_payment = {
    'ApplePay': 'apple_pay',
    'pay pal': 'paypal',
    'SEPA ': 'sepa',
    'googlepay': 'google_pay',
}

df['payment_method'] = df['payment_method'].str.strip().str.lower().replace(mapping_payment)"""

"df.payment_method.unique()\n\nmapping_payment = {\n    'ApplePay': 'apple_pay',\n    'pay pal': 'paypal',\n    'SEPA ': 'sepa',\n    'googlepay': 'google_pay',\n}\n\ndf['payment_method'] = df['payment_method'].str.strip().str.lower().replace(mapping_payment)"

## Début : Suppression temporaire des features catégorielles à conseerver mais non encodés
## Je les rajouterai petit à petit

In [829]:
# Nombre total de lignes
n_rows = len(df)

# Calcul du nombre et du pourcentage de NaN
missing_df = pd.DataFrame({
    'nb_null': df.isnull().sum(),
    'percent_null': df.isnull().sum() / n_rows * 100
})

# Trier par % décroissant
missing_df = missing_df.sort_values(by='percent_null', ascending=False)
missing_df.head(8)

,nb_null,percent_null
region,45296,28.31
age,0,0.00
tenure_months,0,0.00
annual_income_eur,0,0.00
credit_score,0,0.00
num_transactions_30d,0,0.00
avg_amount_30d_eur,0,0.00
max_amount_30d_eur,0,0.00


In [830]:
fill_values = {
    'occupation': 'Missing',
    'merchant_category': 'Missing',
    'last_ticket_subject': 'No_ticket',
    'customer_note': 'No_note'
}

df.fillna(value=fill_values, inplace=True)
df_test.fillna(value=fill_values, inplace=True)

In [831]:
"""cols_to_keep = [
    'chargebacks_12m',
    'ip_risk_z',
    'is_vpn',
    'failed_payments_6m',
    'num_devices_30d',
    'max_amount_30d_eur',
    'tenure_months',
    'support_tickets_90d',
    'target_is_fraud',
    "customer_id"  # à garder uniquement dans le train
]

df = df.drop(columns=[col for col in df.columns if col not in cols_to_keep])"""

'cols_to_keep = [\n    \'chargebacks_12m\',\n    \'ip_risk_z\',\n    \'is_vpn\',\n    \'failed_payments_6m\',\n    \'num_devices_30d\',\n    \'max_amount_30d_eur\',\n    \'tenure_months\',\n    \'support_tickets_90d\',\n    \'target_is_fraud\',\n    "customer_id"  # à garder uniquement dans le train\n]\n\ndf = df.drop(columns=[col for col in df.columns if col not in cols_to_keep])'

# -----------------------------------------------
#                            TRAIN TEST SPLIT
# -----------------------------------------------

In [832]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [833]:
train, val = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['target_is_fraud']
)

# df_test = fichier Kaggle sans target → on le garde tel quel
test = df_test.copy()

In [834]:
# 1. SAUVEGARDER customer_id AVANT TOUT
# ─────────────────────────────────────────
train_customer_ids = train['customer_id']
val_customer_ids = val['customer_id']
test_customer_ids = test['customer_id']

# Retirer customer_id des 3 datasets
train = train.drop(columns=['customer_id'])
val = val.drop(columns=['customer_id'])
test = test.drop(columns=['customer_id'])

In [835]:
# 1. SÉPARATION X ET Y
# On sépare les features de la target pour chaque dataset
# Le modèle a besoin de les avoir séparément
# ─────────────────────────────────────────
X_train = train.drop(columns=['target_is_fraud'])
y_train = train['target_is_fraud']

X_val = val.drop(columns=['target_is_fraud'])
y_val = val['target_is_fraud']

# Test → pas de target (vient de Kaggle)
X_test = test.copy()

Multicolinéarité

In [837]:
"""# Liste des colonnes à supprimer (VIF trop fort)
col_vif_trop_fort = [
    #'terms_accepted_flag',
    'credit_score',
    'avg_amount_30d_eur',
    'age',
]

# Supprimer les colonnes dans train, val et test
for dataset in [train, val, test]:
    dataset.drop(columns=col_vif_trop_fort, inplace=True)"""

"# Liste des colonnes à supprimer (VIF trop fort)\ncol_vif_trop_fort = [\n    #'terms_accepted_flag',\n    'credit_score',\n    'avg_amount_30d_eur',\n    'age',\n]\n\n# Supprimer les colonnes dans train, val et test\nfor dataset in [train, val, test]:\n    dataset.drop(columns=col_vif_trop_fort, inplace=True)"

In [838]:

#device_trust_z → 0
#is_vpn_x_ip_risk → 0
for dataset in [train, val, test]:
    dataset['device_trust_z'] = dataset['device_trust_z'].fillna(0)
    dataset['is_vpn_x_ip_risk'] = dataset['is_vpn_x_ip_risk'].fillna(0)

### encodage des variables catégorielles

In [ ]:
df.occupation

In [848]:
# Colonnes nominales (pas d'ordre) → OneHotEncoder
"""onehot_cols = [
    "signup_source", "os", "browser", "device_type", 
    "channel", "country", "payment_method", 
    "merchant_category", "occupation"
]

# Colonnes ordinales (avec ordre) → OrdinalEncoder
#ordinal_cols = ["plan_type", "manual_review_result"]

ordinal_order = [
    ["basic", "standard", "premium", "enterprise"],  # plan_type
    ["approve", "review", "block"]                    # manual_review_result
]"""

'onehot_cols = [\n    "signup_source", "os", "browser", "device_type", \n    "channel", "country", "payment_method", \n    "merchant_category", "occupation"\n]\n\n# Colonnes ordinales (avec ordre) → OrdinalEncoder\n#ordinal_cols = ["plan_type", "manual_review_result"]\n\nordinal_order = [\n    ["basic", "standard", "premium", "enterprise"],  # plan_type\n    ["approve", "review", "block"]                    # manual_review_result\n]'

### encodage des variables numériques

In [840]:
# Colonnes numériques
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()

# Retirer la target des colonnes numériques
num_cols = [col for col in num_cols if col != 'target_is_fraud']

### création des pipelines pour encodages

In [849]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

onehot_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
"""
ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_order,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])"""

"\nordinal_pipeline = Pipeline(steps=[\n    ('imputer', SimpleImputer(strategy='most_frequent')),\n    ('encoder', OrdinalEncoder(\n        categories=ordinal_order,\n        handle_unknown='use_encoded_value',\n        unknown_value=-1\n    ))\n])"

In [ ]:
# 2. PREPROCESSOR SANS PASSTHROUGH
# On retire le passthrough de la target puisqu'elle est déjà séparée
# remainder='drop' ignore toute colonne non déclarée
# ─────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num',     numeric_pipeline,  num_cols),    # imputation médiane + scaling
    #('onehot',  onehot_pipeline,   onehot_cols), # imputation + OHE
    #('ordinal', ordinal_pipeline,  ordinal_cols), # imputation + OrdinalEncode
])

In [851]:
preprocessor

,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## gerer l'unbalancing sur le train

In [852]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # important ! pas sklearn

In [853]:
# ─────────────────────────────────────────
# 3. PIPELINE SMOTE
# On chaîne le preprocessor et SMOTE dans un seul pipeline
# imblearn Pipeline est nécessaire car sklearn ne supporte pas SMOTE
# ─────────────────────────────────────────
full_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42))
])

In [854]:
X_train.head()

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,failed_payments_6m,device_trust_z,ip_risk_z,is_vpn,num_devices_30d,is_new_device,country,region,terms_accepted_flag,has_second_email,is_missing_max_amount_30d_eur,is_new_device_x_num_devices,is_vpn_x_ip_risk
73786,48,0,48390.11,812,25,76.41,145.74,18,0,0,1,0.080,0.421,0,1,0,FR,Île-de-France,1,0,0,0,0.0
133791,40,3,20920.65,698,21,13.95,47.55,1,1,0,2,-0.799,-0.317,0,4,1,FR,Auvergne-Rhône-Alpes,1,0,0,4,-0.0
79248,25,6,19811.27,756,20,39.89,131.37,23,3,1,1,0.719,-2.020,0,1,0,IT,NaN,1,0,0,0,-0.0
53549,33,17,32596.38,738,22,117.02,361.23,16,1,0,1,1.613,0.231,0,1,0,GB,NaN,1,0,0,0,0.0
19330,55,17,14009.70,708,23,62.85,319.28,6,0,0,0,-0.626,-0.705,0,1,0,BE,NaN,1,0,0,0,-0.0


In [855]:
# 4. FIT + RESAMPLE SUR TRAIN UNIQUEMENT
# fit_resample : 
#   → apprend les statistiques du preprocessor (moyenne, std, catégories...)
#   → transforme X_train
#   → génère des exemples synthétiques avec SMOTE pour équilibrer les classes
# SMOTE ne s'applique JAMAIS sur val et test
# ─────────────────────────────────────────
X_train_res, y_train_res = full_pipeline.fit_resample(X_train, y_train)

In [857]:
# Récupérer les noms de colonnes
#ohe_feature_names = preprocessor.named_transformers_['onehot']['encoder'].get_feature_names_out(onehot_cols).tolist()
#all_cols = num_cols + ohe_feature_names + ordinal_cols

In [858]:
# 5. TRANSFORM SUR VAL ET TEST
# On applique UNIQUEMENT le preprocessor déjà fitté sur train
# Pas de SMOTE ici : on ne rééquilibre pas val et test
# Les statistiques apprises sur train sont appliquées telles quelles
# ─────────────────────────────────────────
X_val_res = preprocessor.transform(X_val)
X_test_res = preprocessor.transform(X_test)

In [859]:
# Recoller X et y avec les bons noms
train_final = pd.DataFrame(X_train_res, columns=num_cols)
train_final['target_is_fraud'] = y_train_res

val_final = pd.DataFrame(X_val_res, columns=num_cols)
val_final.insert(0, 'customer_id', val_customer_ids.values)
val_final['target_is_fraud'] = y_val.values

test_final = pd.DataFrame(X_test_res, columns=num_cols)
test_final.insert(0, 'customer_id', test_customer_ids.values)

In [ ]:
test_final.head()

,customer_id,tenure_months,max_amount_30d_eur,support_tickets_90d,chargebacks_12m,failed_payments_6m,ip_risk_z,is_vpn,num_devices_30d
0,CUST_E5RX1BC9II,1.966287,1.926713,-0.892876,-0.223474,-0.592265,-1.973710,-0.294947,-0.644753
1,CUST_BHWIUKERGN,-0.728610,-0.120455,0.225271,-0.223474,-0.592265,0.438447,-0.294947,-0.644753
2,CUST_EXT9NA4CHU,-0.897041,-0.175294,0.225271,-0.223474,-0.592265,-0.107970,-0.294947,0.483942
3,CUST_9FSJE5R1NY,0.113545,0.144794,-0.892876,-0.223474,1.103604,-0.792515,-0.294947,-0.644753
4,CUST_GDQXMODBED,-0.447892,0.993094,-0.892876,-0.223474,-0.592265,-1.423230,-0.294947,-0.644753


In [860]:
# Exporter
train_final.to_csv('train_1.csv', index=False)
val_final.to_csv('val_1.csv', index=False)
test_final.to_csv('test_1.csv', index=False)